In [18]:
import random
import string
import csv
from PIL import ImageDraw, ImageFont, ImageFilter, Image
import re
from config import *
from pathlib import Path

OUT_DIR = Path("dataset")
IMG_DIR = OUT_DIR / "images"
LABELS = OUT_DIR / "labels.csv"

# spaces are repeated 10x to increase their sampling probability
ALPHABET = string.ascii_uppercase + string.ascii_lowercase + string.digits + " " * 10

WORDS = open("english_words.txt").read().splitlines()

random.seed(2404) # fixed seed for reproducibility

In [20]:
FONTS = [
    "fonts/DejaVuSans.ttf",
    "fonts/DejaVu Sans Bold.ttf",
    "fonts/ARIAL.TTF",
    "fonts/Liberation Sans Regular.ttf"
]

def load_font(size: int):
    """
    Randomly picks a font from the list and loads it at the given size.
    Falls back to default font if the file is not found
    """
    font_path = random.choice(FONTS)
    try:
        return ImageFont.truetype(font_path, size)
    except OSError:
        pass
    return ImageFont.load_default()


def random_word_or_string():
    if random.random() < 0.5:
        # Generates a random string of characters sampled from ALPHABET
        n = random.randint(MIN_LEN, MAX_LEN)
        return "".join(random.choice(ALPHABET) for _ in range(n))
    else:
        # random real words
        k = random.randint(1, 4)
        return " ".join(random.choice(WORDS) for _ in range(k))

def clean_text(s: str) -> str:
    # Strips leading/trailing whitespace and collapses multiple spaces into one
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

def render_image(text: str):
    """
    Renders a grayscale image containing the given text with random augmentations:
    """
    # Random light background (near-white grayscale)
    bg = random.randint(230, 255)
    img = Image.new("L", (IMG_W, IMG_H), bg)
    draw = ImageDraw.Draw(img)

    # Start with font size 24 and shrink if text is too wide
    font_size = 24
    font = load_font(font_size)

    bbox = draw.textbbox((0, 0), text, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    # Reduce font size until text fits within image width
    while text_width > IMG_W - 2 and font_size > 8:
        font_size -= 1
        font = load_font(font_size)
        bbox = draw.textbbox((0, 0), text, font=font)
        text_width = bbox[2] - bbox[0]
        text_height = bbox[3] - bbox[1]

    # Center text horizontally and vertically
    x = max(1, (IMG_W - text_width) // 2 - 4)
    y_jitter = random.randint(-8, 8)
    y = max(0, (IMG_H - text_height) // 2 + y_jitter)

    LETTER_SPACING = random.randint(0,5)

    # Draw characters one by one to get character letter spacing
    x_cursor = x
    for ch in text:
        if random.random() < 0.3:
            for dx in (0, 1):
                for dy in (0, 1):
                    draw.text((x_cursor + dx, y + dy), ch, fill=0, font=font)
        else:
            draw.text((x_cursor, y), ch, fill=0, font=font)

        ch_bbox = draw.textbbox((0, 0), ch, font=font)
        ch_w = ch_bbox[2] - ch_bbox[0]
        x_cursor += ch_w + LETTER_SPACING

    # Augmentation: random per-pixel Gaussian noise
    if random.random() < 0.15:
        px = img.load()
        for _ in range(random.randint(100, 300)):
            x = random.randint(0, IMG_W - 1)
            y = random.randint(0, IMG_H - 1)
            px[x, y] = min(255, max(0, px[x, y] + random.randint(-10, 10)))

    # Augmentation: Gaussian blur
    if random.random() < 0.05:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.2, 0.7)))

    # Augmentation: salt-and-pepper noise (random pure black or white pixels)
    if random.random() < 0.05:
        px = img.load()
        for _ in range(random.randint(10, 40)):
            px[random.randint(0, IMG_W - 1), random.randint(0, IMG_H - 1)] = 0 if random.random() < 0.5 else 255

    # Augmentation: slight rotation (up to +-2 degrees)
    if random.random() < 0.3:
        angle = random.uniform(-2.0, 2.0)
        img = img.rotate(angle, resample=Image.BILINEAR, fillcolor=bg)

    return img

In [21]:
OUT_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

with open(LABELS, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "text"])

    for i in range(N_IMAGES):
        if i % 5000 == 0:
            print(f"Generated {i} images")

        while True:
            text = clean_text(random_word_or_string())
            if text:
                break

        img = render_image(text)

        fname = f"{i:06d}.png"
        img.save(IMG_DIR / fname, optimize=True)
        writer.writerow([fname, text])

print("DONE")
print("Images:", IMG_DIR)
print("Labels:", LABELS)


Generated 0 images
Generated 5000 images
Generated 10000 images
Generated 15000 images
Generated 20000 images
Generated 25000 images
Generated 30000 images
Generated 35000 images
Generated 40000 images
Generated 45000 images
DONE
Images: dataset\images
Labels: dataset\labels.csv
